In [2]:
# .env 파일을 읽어서 환경변수로 설정
from dotenv import load_dotenv

# 토큰 정보로드
load_dotenv()

True

In [3]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install -qU langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH01-Basic")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH01-Basic


## 데이터를 효과적으로 전달하는 방법

- `RunnablePassthrough` 는 입력을 변경하지 않거나 추가 키를 더하여 전달할 수 있습니다. 
- `RunnablePassthrough()` 가 단독으로 호출되면, 단순히 입력을 받아 그대로 전달합니다.
- `RunnablePassthrough.assign(...)` 방식으로 호출되면, 입력을 받아 assign 함수에 전달된 추가 인수를 추가합니다.

### RunnablePassthrough


In [ ]:
from langchain_core.prompts import PromptTemplate

# from langchain_openai import ChatOpenAI
from langchain_upstage import ChatUpstage


# prompt 와 llm 을 생성합니다.
prompt = PromptTemplate.from_template("{num} 의 10배는?")
llm = ChatUpstage(temperature=0)

# chain 을 생성합니다.
chain = prompt | llm

chain 을 `invoke()` 하여 실행할 때는 입력 데이터의 타입이 딕셔너리여야 합니다.

In [5]:
# chain 을 실행합니다.
chain.invoke({"num": 5})

AIMessage(content='5의 10배는 50입니다. 이를 계산하는 방법은 5에 10을 곱하는 것입니다:\n\n\\[ 5 \\times 10 = 50 \\]\n\n따라서, 5의 10배는 50입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 17, 'total_tokens': 78, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'solar-mini-250422', 'system_fingerprint': None, 'id': '673d9281-65aa-4efb-9b2c-4717bd55ed0e', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--b72b09a8-242b-4f30-9f7e-9009d8e8ea07-0', usage_metadata={'input_tokens': 17, 'output_tokens': 61, 'total_tokens': 78, 'input_token_details': {}, 'output_token_details': {}})

하지만, langchain 라이브러리가 업데이트 되면서 1개의 변수만 템플릿에 포함하고 있다면, 값만 전달하는 것도 가능합니다.

In [ ]:
# chain 을 실행합니다.
chain.invoke(5)

아래는 `RunnablePassthrough` 를 사용한 예제입니다.

`RunnablePassthrough` 는 `runnable` 객체이며, `runnable` 객체는 `invoke()` 메소드를 사용하여 별도 실행이 가능합니다.


In [6]:
from langchain_core.runnables import RunnablePassthrough

# runnable
RunnablePassthrough().invoke({"num": 10})

{'num': 10}

In [14]:
from langchain_core.runnables import RunnablePassthrough

answer = RunnablePassthrough()
answer2 = {"num": RunnablePassthrough()}

아래는 `RunnablePassthrough` 로 체인을 구성하는 예제입니다.

In [16]:
answer

RunnablePassthrough()

In [17]:
answer2

{'num': RunnablePassthrough()}

In [22]:
chain = prompt | ChatUpstage()
chain.invoke({"num": 5})

AIMessage(content='5의 10배는 50입니다. 이를 계산하는 방법은 5에 10을 곱하는 것입니다:\n\n\\[ 5 \\times 10 = 50 \\]\n\n따라서, 5의 10배는 50입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 17, 'total_tokens': 78, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'solar-mini-250422', 'system_fingerprint': None, 'id': '81e5a039-0877-45aa-8998-07f87c637d1f', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--6c411f9e-c499-4f93-b24e-506abd50e087-0', usage_metadata={'input_tokens': 17, 'output_tokens': 61, 'total_tokens': 78, 'input_token_details': {}, 'output_token_details': {}})

In [19]:
# prompt는 무조건 딕셔너리를 받는다.
runnable_chain = {"num": RunnablePassthrough()} | prompt | ChatUpstage()

# dict 값이 RunnablePassthrough() 로 변경되었습니다.
runnable_chain.invoke(10)  # 10이 위에 RunnablePassthrough() 로 곧장들어감

AIMessage(content='10의 10배는 100입니다. 이는 10에 10을 곱한 결과입니다. 수학적으로 표현하면:\n\n\\[ 10 \\times 10 = 100 \\]\n\n따라서, 10의 10배는 100입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 19, 'total_tokens': 90, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'solar-mini-250422', 'system_fingerprint': None, 'id': '517dcd47-5da3-4bed-8e2f-e8d42f3e797e', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--68a893d5-330b-4cd4-9511-6b55f804448b-0', usage_metadata={'input_tokens': 19, 'output_tokens': 71, 'total_tokens': 90, 'input_token_details': {}, 'output_token_details': {}})

다음은 `RunnablePassthrough.assign()` 을 사용하는 경우와 비교한 결과입니다.


In [20]:
RunnablePassthrough().invoke({"num": 1})

{'num': 1}

`RunnablePassthrough.assign()`

- 입력 값으로 들어온 값의 key/value 쌍과 새롭게 할당된 key/value 쌍을 합칩니다.

In [ ]:
# 입력 키: num, 할당(assign) 키: new_num
(RunnablePassthrough.assign(new_num=lambda x: x["num"] * 3)).invoke({"num": 1})
# {"num": 1}이 x로 쏙들어감
# x["num"] 키로 값 꺼냄 -> 1
# 1 * 3 = 3
# 3은 new_num 키로 할당됨

{'num': 1, 'new_num': 3}

## RunnableParallel

In [23]:
from langchain_core.runnables import RunnableParallel

# RunnableParallel 인스턴스를 생성합니다. 이 인스턴스는 여러 Runnable 인스턴스를 병렬로 실행할 수 있습니다.
runnable = RunnableParallel(
    # RunnablePassthrough 인스턴스를 'passed' 키워드 인자로 전달합니다. 이는 입력된 데이터를 그대로 통과시키는 역할을 합니다.
    passed=RunnablePassthrough(),
    # 'extra' 키워드 인자로 RunnablePassthrough.assign을 사용하여, 'mult' 람다 함수를 할당합니다. 이 함수는 입력된 딕셔너리의 'num' 키에 해당하는 값을 3배로 증가시킵니다.
    extra=RunnablePassthrough.assign(mult=lambda x: x["num"] * 3),
    # 'modified' 키워드 인자로 람다 함수를 전달합니다. 이 함수는 입력된 딕셔너리의 'num' 키에 해당하는 값에 1을 더합니다.
    modified=lambda x: x["num"] + 1,
)

# runnable 인스턴스에 {'num': 1} 딕셔너리를 입력으로 전달하여 invoke 메소드를 호출합니다.
runnable.invoke({"num": 1})

{'passed': {'num': 1}, 'extra': {'num': 1, 'mult': 3}, 'modified': 2}

Chain 도 RunnableParallel 적용할 수 있습니다.


In [ ]:
chain1 = (
    {"country": RunnablePassthrough()}
    | PromptTemplate.from_template("{country} 의 수도는?")
    | ChatUpstage()
)
chain2 = (
    {"country": RunnablePassthrough()}
    | PromptTemplate.from_template("{country} 의 면적은?")
    | ChatUpstage()
)

In [ ]:
combined_chain = RunnableParallel(capital=chain1, area=chain2)
combined_chain.invoke("대한민국")

## RunnableLambda

RunnableLambda 를 사용하여 사용자 정의 함수를 맵핑할 수 있습니다.


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_upstage import ChatUpstage
from datetime import datetime


def get_today(a):
    # 오늘 날짜를 가져오기
    print(f"입력받은 변수 a의 값: {a}")
    print(f"입력받은 n의 값: {a['n']}")
    a["n"]
    return datetime.today().strftime("%b-%d")


# 오늘 날짜를 출력
# get_today(None)

입력받은 변수 a의 값: None


'Jun-19'

In [47]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from operator import itemgetter

# prompt 와 llm 을 생성합니다.
prompt = PromptTemplate.from_template(
    "{today} 가 생일인 유명인 {n} 명을 나열하세요. 생년월일을 표기해 주세요."
)
llm = ChatUpstage(temperature=0)

# chain 을 생성합니다.
chain = (
    # {"today": RunnableLambda(get_today), "n": RunnablePassthrough()}
    {"today": RunnableLambda(get_today), "n": itemgetter("n")}
    | prompt
    | llm
    | StrOutputParser()
)

In [48]:
# 출력
# print(chain.invoke(3))
print(chain.invoke({"n": 3}))

입력받은 변수 a의 값: {'n': 3}
물론입니다! 다음은 6월 19일에 생일을 맞이하는 유명인 3명입니다:

1. **이민호** (Lee Min-ho) - 1987년 6월 19일 출생. 한국의 인기 배우이자 모델로, 드라마 "꽃보다 남자"와 "도깨비" 등으로 유명합니다.

2. **제시카 차스테인** (Jessica Chastain) - 1977년 6월 19일 출생. 미국의 유명한 배우로, "제로 다크 서티", "인터스텔라", "마더!" 등의 영화에서 주연을 맡았습니다.

3. **제이슨 모모아** (Jason Momoa) - 1979년 6월 19일 출생. 미국의 배우로, "게임 오브 스로운즈"의 칼 드로고 역과 "아쿠아맨"의 주인공 아쿠아맨 역으로 잘 알려져 있습니다.


`itemgetter` 를 사용하여 특정 키를 추출합니다.

In [49]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_upstage import ChatUpstage


# 문장의 길이를 반환하는 함수입니다.
def length_function(text):
    return len(text)


# 두 문장의 길이를 곱한 값을 반환하는 함수입니다.
def _multiple_length_function(text1, text2):
    return len(text1) * len(text2)


# _multiple_length_function 함수를 사용하여 두 문장의 길이를 곱한 값을 반환하는 함수입니다.
def multiple_length_function(_dict):
    return _multiple_length_function(_dict["text1"], _dict["text2"])


prompt = ChatPromptTemplate.from_template("{a} + {b} 는 무엇인가요?")
model = ChatUpstage()

chain1 = prompt | model

chain = (
    {
        "a": itemgetter("word1") | RunnableLambda(length_function),
        "b": {"text1": itemgetter("word1"), "text2": itemgetter("word2")}
        | RunnableLambda(multiple_length_function),
    }
    | prompt
    | model
)

In [50]:
chain.invoke({"word1": "hello", "word2": "world"})

AIMessage(content='5 + 25는 30입니다. \n\n이 문제를 풀기 위해 단순히 두 숫자를 더하면 됩니다:\n\n```\n  5\n+25\n----\n 30\n```\n\n결과적으로 5와 25의 합은 30입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 19, 'total_tokens': 85, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'solar-mini-250422', 'system_fingerprint': None, 'id': '380ea714-90e5-4446-aaac-d93bc16608b6', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--62cbfce3-1f55-44bd-8e2e-65b56e21a2c9-0', usage_metadata={'input_tokens': 19, 'output_tokens': 66, 'total_tokens': 85, 'input_token_details': {}, 'output_token_details': {}})